In [2]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import uuid
import multiprocessing as mp
from multiprocessing import Pool
sc.settings.verbosity = 1

In [3]:
sc.logging.print_header()

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/session_info2/__init__.py:125: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  and (v := getattr(pkg, "__version__", None))
/tmp/ipykernel_1701222/1023403583.py:1: RuntimeWarning: Failed to import dependencies for application/vnd.jupyter.widget-view+json representation. (ModuleNotFoundError: No module named 'ipywidgets')
  sc.logging.print_header()


Package,Version
ipykernel,6.30.1
scipy,1.15.3
numpy,2.2.4
pandas,2.3.1
scanpy,1.11.4
anndata,0.12.1
tqdm,4.67.1
Component,Info
Python,"3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:27:50) [GCC 13.3.0]"
OS,Linux-5.15.0-117-generic-x86_64-with-glibc2.31


## make_anndata, make_anndata_withoutBCR, make_anndata_withoutTCR

In [4]:
def make_anndata(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R3_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index]
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index]
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

In [5]:
def make_anndata_withoutBCR(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R3_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index]
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)

        #Generate new highly_variable assignments except BCR
        bcr = pd.read_table('BCR_gene.txt',header=None)
        for i in range(1,len(bcr.values)):
            ind = adata.var_names.isin(bcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
    
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index]
        
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        #Generate new highly_variable assignments except BCR
        bcr = pd.read_table('BCR_gene.txt',header=None)
        for i in range(1,len(bcr.values)):
            ind = adata.var_names.isin(bcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

In [6]:
def make_anndata_withoutTCR(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R3_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)

        #Generate new highly_variable assignments except TCR
        tcr = pd.read_table('TCR_gene.txt',header=None)
        for i in range(1,len(tcr.values)):
            ind = adata.var_names.isin(tcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()

    
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        #Generate new highly_variable assignments except TCR
        tcr = pd.read_table('TCR_gene.txt',header=None)
        for i in range(1,len(tcr.values)):
            ind = adata.var_names.isin(tcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

# 1. Got HVG and TOTALVI data for L1 refine-low_nCount_RNA

In [7]:
dataset = "low_nCount_RNA"

In [8]:
obj_path = f'/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R3/'

In [11]:
adata_raw = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_{dataset}.h5ad')#600Gb memory, if we add backed='r', use adata = adata_raw[indices.index].to_memory()

In [10]:
adt = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/scADT_MyImmuCell_{dataset}.h5ad')

In [15]:
celltype='CEACAM8_Neg_Neutrophil'

In [16]:
indices = pd.read_csv(f'{obj_path}{celltype}/R3_indices_{celltype}.csv',index_col=0)

In [20]:
sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,
                                subset=True)

In [21]:
##　Save scRNA for TOTALVI
adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")

In [22]:
##　Save scADT for TOTALVI
adt_sub = adt[indices.index].copy()
adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

# 1. Got HVG and TOTALVI data for L1 refine-high_nCount_RNA

In [6]:
dataset = "high_nCount_RNA"

In [7]:
obj_path = f'/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R3/'

In [8]:
#Read in to 1000Gb memory
# if we add backed='r', use adata = adata_raw[indices.index].to_memory(), to process few cell data

adata_raw = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_{dataset}.h5ad',backed='r')

In [9]:
adt = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/scADT_MyImmuCell_{dataset}.h5ad')

## Remove BCR

In [21]:
celltypes=['NaiveB',
           'AtypicalB',
           'Memory_B',]

In [22]:
for i in celltypes:
    make_anndata_withoutBCR(i)

/tmp/ipykernel_4108234/2357668163.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


NaiveB,1036706,1815 Done!


/tmp/ipykernel_4108234/2357668163.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


AtypicalB,79242,1823 Done!


/tmp/ipykernel_4108234/2357668163.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


Memory_B,711801,1797 Done!


## Normal process

In [ ]:
celltypes=[
           'MAIT','gdT',
          ]

In [ ]:
for i in celltypes:
    make_anndata(i)

## Remove TCR

In [ ]:
celltypes=['NaiveCD4',
           'CytotoxicCD4',
           'CD4_helper_memory',
           'Treg',

           'NaiveCD8',
           'CD8Tcm',
           'TemCD8',
           'ProliferativeT',
          ]

In [11]:
celltypes=['DnT',
          ]

In [12]:
for i in celltypes:
    make_anndata_withoutTCR(i)

DnT,7521,1894 Done!


In [10]:
celltypes=[
           'NaiveCD8',
           'TemCD8',
          ]

In [11]:
for i in celltypes:
    make_anndata_withoutTCR(i)

NaiveCD8,1357566,1890 Done!
TemCD8,3960631,1893 Done!
